# Sujet 07 — Santander · Séance 2 « Mesurer honnêtement » · Porteur **P1**
Étapes de P1 (groupe de 3) : **2.1 Le découpage → 2.4 L'étanchéité → 2.7 Le tableau d'essai**.

| Groupe de 3 | Enchaînement | Dans ce carnet |
|---|---|---|
| **P1** | 2.1 → 2.4 → 2.7 | **oui** |
| P2 | 2.2 La barre de bruit → 2.5 Le premier modèle | version minimale (prérequis) |
| P3 | 2.3 Le plancher de contrôle → 2.6 Les comparaisons | 2.3 minimal ; 2.6 absent |

**Les règles de la séance 2.** L'IA est autorisée **à condition de la déclarer** (variable `DECLARATION_IA` dans la configuration). Chaque ligne rendue doit pouvoir être expliquée : l'évaluation porte sur ce que vous savez **refaire sans le carnet et sans l'IA** (lire un code et dire ce qu'il produit, dire ce qui change si quelqu'un modifie une ligne, réécrire une étape de mémoire). Une **fiche de révision** est écrite à la fin dans `sorties/`.

Les cellules **Prérequis** sont des versions minimales des étapes de P2 et P3, juste pour que 2.4 et 2.7 tournent. **Ne les présentez pas comme votre travail.**

**À lancer dans l'ordre, sans sauter de cellule.** Après un redémarrage de session : *Run → Restart session*, puis *Run All*.

## Configuration

In [ ]:
import os, glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import rankdata
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin, clone
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
warnings.filterwarnings("ignore")

# ---------------- PARAMÈTRES ----------------
SEED         = 42      # graine principale
N_SPLITS     = 5       # plis de la validation croisée (modèles rapides)
N_SPLITS_HGB = 3       # plis pour les arbres (plus lents)
HOLDOUT_FRAC = 0.20    # part gardée intouchée, utilisée UNE seule fois
K_LIST       = [1, 5, 10, 50, 100, 200]   # courbe de sélection
ROBUST_SEEDS = [1, 2, 3]                  # graines du test de robustesse
COST_FN      = 10.0    # coût d'une vente manquée (hypothèse, à justifier)
COST_FP      = 1.0     # coût d'une sollicitation inutile (hypothèse)
PLANCHER_ENONCE = 0.5639
# --------------------------------------------

on_kaggle = os.path.exists("/kaggle")
OUT = "/kaggle/working/sorties" if on_kaggle else "sorties"
os.makedirs(OUT, exist_ok=True)

def find(name):
    cands = []
    for root in ["/kaggle/input", "."]:
        if not os.path.exists(root): continue
        for r, d, fs in os.walk(root):
            for f in fs:
                if f.lower() == name.lower():
                    cands.append(os.path.join(r, f))
    cands = [c for c in cands if os.path.getsize(c) > 0]
    if not cands:
        vus = [os.path.join(r, f) for r, d, fs in os.walk("/kaggle/input") for f in fs] if os.path.exists("/kaggle/input") else []
        raise FileNotFoundError(f"{name} introuvable. Fichiers vus dans /kaggle/input : {vus[:20] if vus else 'AUCUN (ajoutez votre jeu de données via Add Input > Your Work)'}")
    return max(cands, key=os.path.getsize)

T0 = time.time()
AUDIT = []
def audit(etape, porteur, titre, constat):
    AUDIT.append(dict(etape=etape, porteur=porteur, titre=titre, constat=constat))
    print(f"[{etape} · {porteur}] {constat}")

# ---------------- ÉQUIPE (à remplir) ----------------
NOM_EQUIPE = "Équipe XX"
MEMBRES    = ["Prénom 1", "Prénom 2", "Prénom 3"]      # <- mettez vos trois prénoms
PORTEURS   = {"P1": MEMBRES[0]}
GROUPE     = 3
COURS      = "ESGC — Sujet 07 Santander"
DATE       = pd.Timestamp.today().strftime("%d/%m/%Y")
RUN_PIPELINE = False    # True : exécute aussi pipeline.py de bout en bout (mesure du temps) ; ajoute quelques minutes
MUT = {}
BASE = "/kaggle/working" if on_kaggle else "."

# ---------------- SÉANCE 2 ----------------
N_TIRAGES = 8      # nombre de tirages pour mesurer le bruit (étape 2.2)
# L'IA est autorisée mais doit être DÉCLARÉE. Complétez avec ce qui est vrai pour votre équipe :
DECLARATION_IA = ("Outil : Claude (Anthropic). Usage : génération du squelette du carnet et des explications. "
                  "Ce que l'équipe a relu, testé et sait refaire sans l'outil : À COMPLÉTER.")

In [ ]:
print(f"{COURS} — {NOM_EQUIPE}\nMembres : {', '.join(MEMBRES)}\nPorteurs : {PORTEURS}\nDate : {DATE}")

In [ ]:
# Diagnostic des données : à lire avant de continuer
vus = []
if os.path.exists("/kaggle/input"):
    for r, d, fs in os.walk("/kaggle/input"):
        for f in fs:
            p = os.path.join(r, f); vus.append((p, os.path.getsize(p)))
for p, t in vus: print(f"{t/1e6:9.1f} Mo  {p}")
if not vus:
    print("AUCUN fichier dans /kaggle/input : ajoutez votre jeu de données (Add Input > Your Work > le +) ou, en local, mettez les CSV à côté du carnet.")
elif max(t for _, t in vus) < 1_000_000:
    print("ATTENTION : tous les fichiers pèsent moins de 1 Mo. Ce ne sont pas les vrais fichiers (train.csv doit faire environ 250 Mo). Recréez le jeu de données.")

## Boîte à outils
Définitions seulement.

In [ ]:
def uni_auc(X, y):
    # AUC de chaque colonne seule, vectorisée (rangs)
    r = rankdata(X, axis=0)
    n1 = y.sum(); n0 = len(y) - n1
    return (r[y == 1].sum(axis=0) - n1*(n1+1)/2) / (n1*n0)

def score(model, X):
    # score continu : logit de la proba (sert à l'AUC et à la calibration)
    p = np.clip(model.predict_proba(X)[:, 1], 1e-9, 1-1e-9)
    return np.log(p/(1-p))

def cv_eval(model, X, y, n_splits=N_SPLITS, seed=SEED):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y)); aucs = []
    t = time.time()
    for tr, va in skf.split(X, y):
        m = clone(model).fit(X[tr], y[tr])
        oof[va] = score(m, X[va])
        aucs.append(roc_auc_score(y[va], oof[va]))
    return dict(oof=oof, mean=np.mean(aucs), std=np.std(aucs), time=time.time()-t)

JOURNAL = []
def log(nom, res, note="", k=None):
    JOURNAL.append(dict(essai=nom, k=k, auc=round(res["mean"], 5), ecart_type=round(res["std"], 5),
                        secondes=round(res["time"], 1), note=note))
    print(f"{nom:<38} AUC {res['mean']:.5f} ± {res['std']:.5f}  ({res['time']:.0f} s)  {note}")

In [ ]:
class TopK(BaseEstimator, TransformerMixin):
    # garde les k variables dont l'AUC seule est la plus éloignée de 0,5, calculée sur le pli d'entraînement
    def __init__(self, k=200): self.k = k
    def fit(self, X, y):
        a = uni_auc(np.asarray(X), np.asarray(y))
        self.idx_ = np.argsort(-np.abs(a - 0.5))[:self.k]
        return self
    def transform(self, X): return np.asarray(X)[:, self.idx_]

class NBBin(BaseEstimator, ClassifierMixin):
    # Naive Bayes par variable discrétisée en quantiles : somme de rapports de vraisemblance (modèle additif)
    def __init__(self, n_bins=20, alpha=1.0): self.n_bins = n_bins; self.alpha = alpha
    def fit(self, X, y):
        X = np.asarray(X); y = np.asarray(y); self.classes_ = np.array([0, 1])
        qs = np.linspace(0, 1, self.n_bins + 1)[1:-1]
        self.edges_ = [np.unique(np.quantile(X[:, j], qs)) for j in range(X.shape[1])]
        self.llr_ = []
        for j, e in enumerate(self.edges_):
            b = np.searchsorted(e, X[:, j]); nb = len(e) + 1
            c1 = np.bincount(b[y == 1], minlength=nb) + self.alpha
            c0 = np.bincount(b[y == 0], minlength=nb) + self.alpha
            self.llr_.append(np.log(c1/c1.sum()) - np.log(c0/c0.sum()))
        self.prior_ = np.log(y.mean()/(1 - y.mean()))
        return self
    def decision_function(self, X):
        X = np.asarray(X); s = np.full(len(X), self.prior_)
        for j, e in enumerate(self.edges_): s += self.llr_[j][np.searchsorted(e, X[:, j])]
        return s
    def predict_proba(self, X):
        p = 1/(1 + np.exp(-self.decision_function(X))); return np.c_[1 - p, p]

class FreqAdd(BaseEstimator, TransformerMixin):
    # ajoute, pour chaque variable, log(1 + nb d'apparitions de la valeur arrondie) ; comptage appris sur le pli
    def __init__(self, decimals=4): self.decimals = decimals
    def fit(self, X, y=None):
        X = np.asarray(X)
        self.counts_ = [pd.Series(np.round(X[:, j], self.decimals)).value_counts() for j in range(X.shape[1])]
        return self
    def transform(self, X):
        X = np.asarray(X)
        F = np.column_stack([np.log1p(pd.Series(np.round(X[:, j], self.decimals)).map(self.counts_[j]).fillna(0).values)
                            for j in range(X.shape[1])])
        return np.hstack([X, F])

def hgb(depth=3, lr=0.1, it=200, **kw):
    return HistGradientBoostingClassifier(max_depth=depth, learning_rate=lr, max_iter=it,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=SEED, **kw)
def logreg(C=0.1): return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=300))

## Chargement des données *(prérequis, commun à tous les porteurs)*

In [ ]:
FILES = {n: find(n) for n in ["train.csv", "a_decider.csv"]}
for n, p in FILES.items():
    t = os.path.getsize(p); print(f"{n:<14} {p}  ({t/1e6:.1f} Mo)")
    if t < 1_000_000: raise RuntimeError(f"{n} ne pèse que {t} octets : pas le vrai fichier. Recréez le jeu de données.")
train = pd.read_csv(FILES["train.csv"]); test = pd.read_csv(FILES["a_decider.csv"])
FEATS = [c for c in train.columns if c.startswith("var_")]
X_all = train[FEATS].values.astype(np.float64); y_all = train["target"].values.astype(int); ID_all = train["ID_code"].astype(str).values
print("train", train.shape, "| a_decider", test.shape, "| variables :", len(FEATS), "| positifs : %.2f %%" % (100 * y_all.mean()))
print("Conforme à l'énoncé (160000 x 202 / 40000 x 201) :", train.shape == (160000, 202) and test.shape == (40000, 201))

## 2.1 Le découpage · *P1*
**Ce que demande la séance :** diviser les données en une partie qui **apprend** et une partie qui **mesure** ; écrire le découpage choisi, la raison, et le nombre de parties.

**Décision.**
- **Deux grandes parties** : `dev` (80 %) pour apprendre et comparer, `holdout` (20 %) pour contrôler **une seule fois** à la fin.
- Dans `dev`, **5 plis** de validation croisée pour mesurer pendant les essais. **Nombre de parties : 6** (5 plis + holdout).
- **Aléatoire**, parce que les clients sont indépendants et sans dimension temporelle ; **stratifié**, pour garder ~10 % de positifs dans chaque partie ; **graine fixée**, pour que toute l'équipe ait le même découpage (`decoupage.csv`).

**À savoir refaire sans le carnet :** `train_test_split(..., test_size=0.2, stratify=y, random_state=42)` pour `dev`/`holdout`, puis `StratifiedKFold(5, shuffle=True, random_state=42)` sur `dev`.

In [ ]:
idx_all = np.arange(len(y_all))
idx_dev, idx_hold = train_test_split(idx_all, test_size=HOLDOUT_FRAC, stratify=y_all, random_state=SEED)
pli = np.full(len(y_all), -1)                     # -1 = holdout ; 0..N_SPLITS-1 = pli de validation (dans dev)
skf = StratifiedKFold(N_SPLITS, shuffle=True, random_state=SEED)
for k, (tr, va) in enumerate(skf.split(idx_dev, y_all[idx_dev])):
    pli[idx_dev[va]] = k
X_dev, y_dev = X_all[idx_dev], y_all[idx_dev]
X_hold, y_hold = X_all[idx_hold], y_all[idx_hold]

# contrôles du découpage
assert len(set(idx_dev) & set(idx_hold)) == 0, "dev et holdout se recouvrent !"
rows = [("holdout", len(idx_hold), y_all[idx_hold].mean())] + [(f"pli {k}", int((pli == k).sum()), y_all[pli == k].mean()) for k in range(N_SPLITS)]
tab = pd.DataFrame(rows, columns=["partie", "lignes", "taux de positifs"]); tab["taux de positifs"] = (100 * tab["taux de positifs"]).round(2).astype(str) + " %"
display(tab)
ecart = max(abs(y_all[pli == k].mean() - y_all.mean()) for k in range(-1, N_SPLITS))
print("Taux global : %.2f %% | plus grand écart d'une partie : %.3f point" % (100 * y_all.mean(), 100 * ecart))
pd.DataFrame({"ID_code": ID_all, "partie": np.where(pli < 0, "holdout", "dev"), "pli": pli}).to_csv(f"{OUT}/decoupage.csv", index=False)
print("Découpage enregistré :", f"{OUT}/decoupage.csv", "(à partager avec P2 et P3)")
audit("2.1", "P1", "Découpage", f"holdout {len(idx_hold)} ({100*HOLDOUT_FRAC:.0f} %), dev {len(idx_dev)} en {N_SPLITS} plis stratifiés, graine {SEED} ; écart max de taux {100*ecart:.3f} pt ; dev/holdout disjoints")

## Prérequis — 2.2 La barre de bruit et 2.3 Le plancher de contrôle *(P2 et P3, version minimale)*
- **2.2 Barre de bruit** : on refait la **même mesure** (AUC de la régression logistique) en changeant **seulement le tirage** de la séparation apprentissage/contrôle dans `dev`. La dispersion obtenue est exprimée en **points d'AUC**. La barre de bruit est prise à **2 écarts-types** : un écart plus petit n'est pas distinguable du hasard.
- **2.3 Plancher de contrôle** : une règle qui **n'utilise aucune variable** : « classe majoritaire » (répondre toujours « non »), mesurée sur les parties de contrôle. À côté, le **plancher métier** : la meilleure variable seule (l'énoncé donne 0,5639).

In [ ]:
# --- 2.2 : même mesure, on change seulement le tirage ---
aucs_tirages = []
for s in range(N_TIRAGES):
    tr, va = train_test_split(np.arange(len(y_dev)), test_size=0.2, stratify=y_dev, random_state=1000 + s)
    m = logreg().fit(X_dev[tr], y_dev[tr])
    aucs_tirages.append(roc_auc_score(y_dev[va], score(m, X_dev[va])))
DISPERSION = float(np.std(aucs_tirages, ddof=1)); BARRE_BRUIT = 2 * DISPERSION
print("AUC sur %d tirages : min %.4f | max %.4f | moyenne %.4f" % (N_TIRAGES, min(aucs_tirages), max(aucs_tirages), np.mean(aucs_tirages)))
print("Dispersion (écart-type) : %.4f point d'AUC | barre de bruit (2 écarts-types) : ±%.4f" % (DISPERSION, BARRE_BRUIT))

# --- 2.3 : règle sans aucune variable = classe majoritaire, apprise sur l'entraînement de chaque pli ---
acc_pli = []
for k in range(N_SPLITS):
    tr = np.where((pli >= 0) & (pli != k))[0]; va = np.where(pli == k)[0]
    classe_maj = int(round(y_all[tr].mean()))            # 0 : "non" est la classe majoritaire
    acc_pli.append(float((y_all[va] == classe_maj).mean()))
PLANCHER_CONTROLE = roc_auc_score(y_dev, np.zeros(len(y_dev)))   # score constant : AUC = 0,5
ACC_MAJ = float(np.mean(acc_pli))
a_dev = uni_auc(X_dev, y_dev); auc_seule = np.maximum(a_dev, 1 - a_dev)
PLANCHER_METIER = auc_seule.max(); VAR_PLANCHER = FEATS[int(np.argmax(auc_seule))]
print("Plancher de contrôle : règle = répondre toujours « non » | exactitude %.2f %% | AUC %.4f" % (100 * ACC_MAJ, PLANCHER_CONTROLE))
print("Plancher métier mesuré : %s, AUC %.4f (énoncé : %s)" % (VAR_PLANCHER, PLANCHER_METIER, PLANCHER_ENONCE))

## 2.4 L'étanchéité · *P1* (attend 2.1)
**Ce que demande la séance :** pour **chaque traitement** (remplir un trou, encoder une catégorie, transformer une variable…), dire **sur quelle partie des données il est appris**. Livrable : la liste des traitements avec cette partie.

La cellule ci-dessous commence par cette **liste**. Elle ajoute ensuite des **vérifications complémentaires** (plis disjoints, doublons, test sur étiquettes mélangées) pour prouver que la liste est bien respectée.

**À savoir refaire sans le carnet :** pour chaque traitement, la phrase « calculé sur la partie d'entraînement du pli, puis appliqué à la validation ». Ce qui change si on le calcule sur tout `dev` avant les plis : la validation « voit » des informations de l'entraînement, le score est trop beau.

In [ ]:
def cv_pli(model, X, y, pli, y_fit=None):
    yf = y if y_fit is None else y_fit
    oof = np.full(len(y), np.nan); aucs = []; t = time.time()
    for k in range(N_SPLITS):
        tr = np.where((pli >= 0) & (pli != k))[0]; va = np.where(pli == k)[0]
        m = clone(model).fit(X[tr], yf[tr]); s = score(m, X[va]); oof[va] = s
        aucs.append(roc_auc_score(yf[va], s))
    return dict(oof=oof, mean=float(np.mean(aucs)), std=float(np.std(aucs)), time=time.time() - t, aucs=aucs)

# ===== A. LA LISTE DES TRAITEMENTS (livrable de l'étape 2.4) =====
TRAITEMENTS = pd.DataFrame([
    ("Découpage dev / holdout / 5 plis", "oui (2.1)", "décidé avant tout traitement ; il n'utilise que la cible pour stratifier"),
    ("Remplissage des trous", "non : aucune valeur manquante (étape 1.6)", "sans objet"),
    ("Encodage de catégories", "non : aucune variable catégorielle", "sans objet"),
    ("Standardisation (moyenne, écart-type)", "oui, dans le modèle de référence", "partie d'entraînement de CHAQUE pli seulement (dans le Pipeline), puis appliquée à la validation"),
    ("Règle « classe majoritaire » (plancher de contrôle)", "oui (2.3)", "partie d'entraînement de chaque pli"),
    ("Sélection de variables (si utilisée)", "pas dans le premier modèle", "partie d'entraînement du pli (jamais sur tout dev)"),
    ("Quantiles / fréquences (si utilisés)", "pas dans le premier modèle", "partie d'entraînement du pli"),
], columns=["traitement", "utilisé dans ce projet ?", "appris sur quelle partie ?"])
pd.set_option("display.max_colwidth", 110); pd.set_option("display.width", 220)
display(TRAITEMENTS)

# ===== B. VÉRIFICATIONS COMPLÉMENTAIRES =====
pi = y_all.mean()
# 1. disjonction
ok_disj = all(len(set(np.where((pli >= 0) & (pli != k))[0]) & set(np.where(pli == k)[0])) == 0 for k in range(N_SPLITS)) and len(set(idx_dev) & set(idx_hold)) == 0
print("1. Entraînement/validation disjoints dans chaque pli, dev/holdout disjoints :", ok_disj)
# 2. doublons dev / holdout
h_dev = pd.util.hash_pandas_object(pd.DataFrame(X_dev), index=False).values; h_hold = pd.util.hash_pandas_object(pd.DataFrame(X_hold), index=False).values
n_dup = int(np.isin(h_hold, h_dev).sum())
print("2. Lignes du holdout identiques à une ligne de dev :", n_dup)
# 3. étiquettes mélangées sur un petit échantillon : sélection DANS le pli (propre) contre AVANT le découpage (fuite volontaire)
n_sub = min(3000, len(idx_dev)); rr = np.random.RandomState(SEED + 1)
clean, leaky = [], []
for rep in range(10):
    sub = rr.choice(len(idx_dev), n_sub, replace=False); Xs = X_dev[sub]; ys = rr.permutation(y_dev[sub])
    sk = StratifiedKFold(N_SPLITS, shuffle=True, random_state=rep)
    pc = np.zeros(n_sub); pl = np.zeros(n_sub)
    sel_leak = np.argsort(-np.abs(uni_auc(Xs, ys) - 0.5))[:10]            # FUITE : sélection avec toutes les étiquettes
    for tr, va in sk.split(Xs, ys):
        pc[va] = score(make_pipeline(TopK(10), logreg()).fit(Xs[tr], ys[tr]), Xs[va])      # propre : sélection dans le pli
        pl[va] = score(logreg().fit(Xs[tr][:, sel_leak], ys[tr]), Xs[va][:, sel_leak])      # fuite
    clean.append(roc_auc_score(ys, pc)); leaky.append(roc_auc_score(ys, pl))
ok_clean = abs(np.mean(clean) - 0.5) < 0.03
sensible = np.mean(leaky) > np.mean(clean) + 0.02
print("3. Étiquettes mélangées (aucun signal), %d lignes : sélection DANS le pli AUC %.3f (attendu ≈ 0,5) | sélection AVANT le découpage (fuite) AUC %.3f" % (n_sub, np.mean(clean), np.mean(leaky)))
print("   -> sélection propre :", ok_clean, "| le test détecte bien une fuite :", sensible)
# 4. pipeline complet sur étiquettes mélangées
y_perm = y_all.copy(); di = np.where(pli >= 0)[0]; y_perm[di] = np.random.RandomState(SEED + 2).permutation(y_all[di])
res_perm = cv_pli(logreg(), X_all, y_all, pli, y_fit=y_perm)
tol_full = max(BARRE_BRUIT, 0.01)
ok_full = abs(res_perm["mean"] - 0.5) < tol_full
print("4. Pipeline complet, étiquettes mélangées : AUC %.4f (attendu 0,5 ± %.4f) -> %s" % (res_perm["mean"], tol_full, ok_full))
print("5. Holdout non utilisé dans ce carnet (règle à respecter par toute l'équipe).")
ETANCHE = bool(ok_disj and n_dup == 0 and ok_clean and ok_full)
print("\nVERDICT D'ÉTANCHÉITÉ :", "ÉTANCHE" if ETANCHE else "FUITE POSSIBLE : À EXAMINER avant tout essai")
audit("2.4", "P1", "Étanchéité", f"{len(TRAITEMENTS)} traitements listés avec leur partie d'apprentissage ; disjoint {ok_disj} ; doublons {n_dup} ; sélection dans le pli {np.mean(clean):.3f} vs fuite {np.mean(leaky):.3f} ; pipeline sur étiquettes mélangées {res_perm['mean']:.4f} ; verdict : {'étanche' if ETANCHE else 'à examiner'}")

## Prérequis — 2.5 Le premier modèle *(étape de P2, version minimale)*
Régression logistique sur les 200 variables standardisées, avec **le même découpage** que 2.1.

In [ ]:
res_first = cv_pli(logreg(), X_all, y_all, pli)
print("Premier modèle (régression logistique, 200 var.) : AUC %.5f ± %.5f (%.0f s)" % (res_first["mean"], res_first["std"], res_first["time"]))
print("Gain sur le plancher métier de l'énoncé (%.4f) : %+.4f | barre de bruit ±%.4f" % (PLANCHER_ENONCE, res_first["mean"] - PLANCHER_ENONCE, BARRE_BRUIT))

## 2.7 Le tableau d'essai · *P1*
**Ce que demande la séance :** écrire la ligne du tableau d'essais : **la date, les variables utilisées, le score, et si le plancher est battu.**

Un plancher est « battu » seulement si le gain dépasse la **barre de bruit** (étape 2.2) : un écart plus petit peut venir du hasard du tirage. Les essais de P2 et P3 s'ajoutent avec `ajouter_essai(...)`. **Rien n'est inventé** : une ligne n'existe que si elle a été mesurée.

**À savoir refaire sans le carnet :** une ligne = `date | variables | score | plancher battu ?`, et la règle « gain > barre de bruit ».

In [ ]:
ESSAIS = []
DEC = f"{N_SPLITS} plis stratifiés, graine {SEED}, holdout {int(100*HOLDOUT_FRAC)} % intouché"
AUJ = pd.Timestamp.today().strftime("%Y-%m-%d")
def ajouter_essai(etape, nom, variables, auc, std, temps, auteur):
    ESSAIS.append(dict(date=AUJ, etape=etape, essai=nom, variables=variables, découpage=DEC, auc=round(float(auc), 5),
                       ecart_type=round(float(std), 5), secondes=round(float(temps), 1), auteur=auteur))

ajouter_essai("2.3", "Plancher de contrôle (classe majoritaire)", "aucune", PLANCHER_CONTROLE, 0, 0, "P3")
ajouter_essai("2.3", "Plancher métier", f"1 variable : {VAR_PLANCHER}", PLANCHER_METIER, 0, 0, "P3")
ajouter_essai("2.5", "Premier modèle : régression logistique", "200 variables (var_0 à var_199), standardisées", res_first["mean"], res_first["std"], res_first["time"], "P2")
ajouter_essai("2.4", "Contrôle d'étanchéité (étiquettes mélangées)", "200 variables, étiquettes mélangées", res_perm["mean"], res_perm["std"], res_perm["time"], "P1")

# --- Les essais des autres porteurs (2.6 Les comparaisons) : décommentez et complétez avec VOS mesures ---
# ajouter_essai("2.6", "NB discrétisé", "200 variables", 0.0, 0.0, 0.0, "P3")

T = pd.DataFrame(ESSAIS)
T["gain_vs_controle"] = (T.auc - PLANCHER_CONTROLE).round(5)
T["gain_vs_plancher_enonce"] = (T.auc - PLANCHER_ENONCE).round(5)
def bat(g, r):
    if r.etape == "2.3": return "plancher"
    if r.etape == "2.4": return "contrôle : doit rester ≈ 0,5"
    return "oui" if g > BARRE_BRUIT else "non (dans le bruit)"
T["bat_plancher_controle"] = [bat(r.gain_vs_controle, r) for r in T.itertuples()]
T["bat_plancher_enonce_0.5639"] = [bat(r.gain_vs_plancher_enonce, r) for r in T.itertuples()]
cols = ["date", "etape", "essai", "variables", "auc", "ecart_type", "secondes", "bat_plancher_controle", "bat_plancher_enonce_0.5639", "auteur"]
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 240)
display(T[cols])
T.to_csv(f"{OUT}/tableau_essais.csv", index=False)
with open(f"{OUT}/tableau_essais.md", "w") as f:
    f.write(f"# Tableau d'essai — {NOM_EQUIPE} ({DATE})\n\nDispersion (2.2) : {DISPERSION:.4f} ; barre de bruit : ±{BARRE_BRUIT:.4f}. Étanchéité : {'étanche' if ETANCHE else 'À EXAMINER'}.\n\n")
    f.write("| date | étape | essai | variables | AUC | écart-type | s | bat le plancher de contrôle | bat 0,5639 | auteur |\n|---|---|---|---|---|---|---|---|---|---|\n")
    for _, r in T.iterrows(): f.write(f"| {r.date} | {r.etape} | {r.essai} | {r.variables} | {r.auc} | {r.ecart_type} | {r.secondes} | {r.bat_plancher_controle} | {r['bat_plancher_enonce_0.5639']} | {r.auteur} |\n")
print("Écrit :", f"{OUT}/tableau_essais.csv et .md", "| essais consignés :", len(T), "(l'énoncé en demande au moins 8 avant la séance 9)")
audit("2.7", "P1", "Tableau d'essai", f"{len(T)} essais consignés (date, variables, score, plancher battu) ; barre de bruit ±{BARRE_BRUIT:.4f}")

## Récapitulatif de P1 et fichier de résultats

In [ ]:
manquants = [n for n in ["pli", "idx_dev", "idx_hold", "ETANCHE", "res_first", "res_perm", "T", "BARRE_BRUIT", "DISPERSION", "clean", "leaky", "n_dup", "ok_disj", "TRAITEMENTS"] if n not in globals()]
if manquants:
    raise RuntimeError("Étapes non exécutées : " + ", ".join(manquants) + ". Faites Run > Restart session puis Run All depuis le haut.")
A = pd.DataFrame(AUDIT); A.insert(2, "responsable", A.porteur.map(PORTEURS).fillna(PORTEURS.get("P1")))
pd.set_option("display.max_colwidth", 190); display(A)
trait_md = "\n".join(f"| {r[0]} | {r[1]} | {r[2]} |" for r in TRAITEMENTS.itertuples(index=False))
R = f"""# Résultats du porteur P1, séance 2 — {COURS}
**{NOM_EQUIPE}** · P1 : {PORTEURS['P1']} · {DATE}

## Déclaration d'usage de l'IA
{DECLARATION_IA}

## 2.1 Le découpage
- **Deux parties** : dev ({len(idx_dev)} lignes) pour apprendre, holdout ({len(idx_hold)} lignes, {int(100*HOLDOUT_FRAC)} %) pour contrôler une seule fois. Dans dev : {N_SPLITS} plis. **Nombre de parties : {N_SPLITS + 1}.**
- **Raison** : clients indépendants, pas de dimension temporelle -> découpage aléatoire ; stratifié pour garder {100*y_all.mean():.2f} % de positifs partout (écart maximal observé : {100*ecart:.3f} point) ; graine {SEED} pour que l'équipe partage le même découpage (`decoupage.csv`).

## 2.4 L'étanchéité
### Liste des traitements et partie sur laquelle chacun est appris
| Traitement | Utilisé ? | Appris sur |
|---|---|---|
{trait_md}

### Vérifications complémentaires
| Contrôle | Résultat |
|---|---|
| Entraînement/validation disjoints (chaque pli), dev/holdout disjoints | {ok_disj} |
| Lignes du holdout identiques à du dev | {n_dup} |
| Étiquettes mélangées, sélection DANS le pli (attendu ≈ 0,5) | {np.mean(clean):.3f} |
| Étiquettes mélangées, sélection AVANT le découpage (fuite volontaire) | {np.mean(leaky):.3f} |
| Pipeline complet sur étiquettes mélangées (attendu ≈ 0,5) | {res_perm['mean']:.4f} |

**Verdict : {'ÉTANCHE' if ETANCHE else 'FUITE POSSIBLE, À EXAMINER avant tout essai'}.**

## 2.7 Le tableau d'essai
Voir `tableau_essais.md` : {len(T)} essais consignés (date, variables, score, plancher battu). Dispersion mesurée en 2.2 : {DISPERSION:.4f} point d'AUC ; barre de bruit ±{BARRE_BRUIT:.4f}. Un plancher n'est « battu » que si le gain dépasse cette barre.
"""
open(f"{OUT}/resultats_P1_seance2.md", "w").write(R); print(R)

FICHE = """# Fiche de révision P1 — à savoir refaire SANS le carnet et SANS l'IA
(à relire, puis à réécrire de mémoire sur papier)

## 2.1 Le découpage
Refaire : 1) `train_test_split(test_size=0.2, stratify=y, random_state=42)` -> dev / holdout ;
2) `StratifiedKFold(5, shuffle=True, random_state=42)` sur dev ; 3) vérifier ~10 % de positifs dans chaque partie.
Si on change une ligne :
- on enlève `stratify` -> le taux de positifs varie d'une partie à l'autre ;
- on change `random_state` -> autre découpage, score un peu différent (c'est le bruit, étape 2.2) ;
- on passe `test_size` à 0.5 -> moins de données pour apprendre, mesure moins stable ;
- on utilise le holdout pour choisir un modèle -> il n'est plus un vrai contrôle.

## 2.4 L'étanchéité
Refaire : pour CHAQUE traitement, écrire la partie sur laquelle il est appris.
Exemple : la standardisation calcule moyenne et écart-type sur l'entraînement du pli, puis s'applique à la validation.
Si on change une ligne :
- on standardise ou on sélectionne les variables sur tout dev avant les plis -> la validation « voit » l'entraînement, le score est trop beau ;
- sur des étiquettes mélangées (aucun signal) la bonne méthode donne AUC ~ 0,5 ; la sélection faite avant le découpage donne plus que 0,5 -> c'est une fuite.

## 2.7 Le tableau d'essai
Refaire : une ligne = date | variables utilisées | score | plancher battu ?
Règle : un plancher est battu seulement si le gain est plus grand que la barre de bruit (2 écarts-types de la dispersion mesurée en 2.2).
Si on change une ligne :
- on écrit un score non mesuré -> ligne inventée, interdit ;
- le gain est plus petit que la barre de bruit -> « non, dans le bruit ».
"""
open(f"{OUT}/fiche_revision_P1.md", "w").write(FICHE)
print("Écrit :", f"{OUT}/resultats_P1_seance2.md", "et", f"{OUT}/fiche_revision_P1.md")